In [13]:
import os

os.listdir()

['.config', 'luad_tcga_pub.tar.gz', 'sample_data']

In [14]:
!tar -xzf luad_tcga_pub.tar.gz

In [15]:
!ls

luad_tcga_pub  luad_tcga_pub.tar.gz  sample_data


In [16]:
import os

for root, dirs, files in os.walk("."):
    for f in files:
        if "mutation" in f.lower():
            print(os.path.join(root, f))

./luad_tcga_pub/meta_mutations.txt
./luad_tcga_pub/data_mutations.txt


In [17]:
import pandas as pd

df = pd.read_csv(
    "./luad_tcga_pub/data_mutations.txt",
    sep="\t",
    comment="#",
    low_memory=False
)

print(df.shape)
df.head()

(72566, 250)


,Hugo_Symbol,Entrez_Gene_Id,Center,NCBI_Build,Chromosome,Start_Position,End_Position,Strand,Consequence,Variant_Classification,...,SWISSPROT,ExAC_AF_FIN,cgc_translocation_partner,HGVS_OFFSET,PolyPhen,refseq_prot_id,Annotation_Transcript,SAS_MAF,oxoGCut,MINIMISED
0,PLEKHN1,84069,broad.mit.edu,GRCh37,1,905907,905907,+,missense_variant,Missense_Mutation,...,PKHN1_HUMAN,NaN,NaN,NaN,probably_damaging(1),NP_115505,uc001ace.2,NaN,0.0,1.0
1,UBE2J2,118424,broad.mit.edu,GRCh37,1,1192480,1192480,+,synonymous_variant,Silent,...,UB2J2_HUMAN,NaN,NaN,NaN,NaN,NP_919440,uc001adn.2,NaN,0.0,NaN
2,C1orf222,85452,broad.mit.edu,GRCh37,1,1854885,1854885,+,missense_variant,Missense_Mutation,...,NaN,NaN,NaN,NaN,probably_damaging(0.987),NaN,uc001aik.2,NaN,0.0,NaN
3,C1orf200,644997,broad.mit.edu,GRCh37,1,9713992,9713992,+,missense_variant,Missense_Mutation,...,CA200_HUMAN,NaN,NaN,NaN,possibly_damaging(0.835),NaN,uc001aqc.3,NaN,0.0,NaN
4,HNRNPCL1,343069,broad.mit.edu,GRCh37,1,12908093,12908093,+,missense_variant,Missense_Mutation,...,HNRCL_HUMAN,NaN,NaN,NaN,possibly_damaging(0.491),NP_001139653,uc009vno.2,NaN,0.0,NaN


In [18]:
print(df.columns.tolist())

['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons', 'Hotspot', 'i_init_t_lod', 'ALLELE_NUM', 'PICK', 'mutsig_published_results', 'MA:link.PDB', 'Feature', 'Transcript_Position', 'ref_allele', 'HGNC_ID', 'ExAC_AF_AMR', 'SwissProt_entry_Id', 'qox', 'M

In [19]:
print("Rows:", len(df))
print("Genes:", df["Hugo_Symbol"].nunique())
print("Patients:", df["Tumor_Sample_Barcode"].nunique())

Rows: 72566
Genes: 15130
Patients: 230


In [20]:
df["Hugo_Symbol"].value_counts().head(20)

,count
Hugo_Symbol,
TTN,363
MUC16,205
RYR2,162
CSMD3,149
LRP1B,138
USH2A,124
TP53,116
ZFHX4,114
FLG,98


In [21]:
patient_gene = pd.crosstab(
    df["Tumor_Sample_Barcode"],
    df["Hugo_Symbol"]
)

patient_gene = (patient_gene > 0).astype(int)

patient_gene.shape

(230, 15130)

In [22]:
gene_counts = patient_gene.sum(axis=0)

selected_genes = gene_counts[
    gene_counts >= 5
].index

patient_gene_small = patient_gene[selected_genes]

patient_gene_small.shape

(230, 4730)

In [23]:
co_matrix = patient_gene_small.T.dot(
    patient_gene_small
)

co_matrix.iloc[:5,:5]

Hugo_Symbol,A1CF,A2M,A2ML1,A4GNT,AACS
Hugo_Symbol,,,,,
A1CF,7,1,1,0,0
A2M,1,11,0,0,0
A2ML1,1,0,9,1,0
A4GNT,0,0,1,5,0
AACS,0,0,0,0,6


In [24]:
import numpy as np

pairs = []

for i in range(len(co_matrix.columns)):
    for j in range(i+1, len(co_matrix.columns)):

        gene1 = co_matrix.columns[i]
        gene2 = co_matrix.columns[j]

        count = co_matrix.iloc[i,j]

        if count >= 10:
            pairs.append(
                (gene1,gene2,count)
            )

pairs = sorted(
    pairs,
    key=lambda x:x[2],
    reverse=True
)

pairs[:20]

[('TP53', 'TTN', np.int64(74)),
 ('MUC16', 'TTN', np.int64(72)),
 ('RYR2', 'TTN', np.int64(68)),
 ('CSMD3', 'TTN', np.int64(66)),
 ('LRP1B', 'TTN', np.int64(62)),
 ('CSMD3', 'TP53', np.int64(59)),
 ('CSMD3', 'MUC16', np.int64(57)),
 ('MUC16', 'RYR2', np.int64(57)),
 ('MUC16', 'TP53', np.int64(57)),
 ('RYR2', 'TP53', np.int64(57)),
 ('TTN', 'USH2A', np.int64(57)),
 ('LRP1B', 'RYR2', np.int64(56)),
 ('TTN', 'ZFHX4', np.int64(56)),
 ('CSMD3', 'LRP1B', np.int64(53)),
 ('CSMD3', 'RYR2', np.int64(53)),
 ('LRP1B', 'MUC16', np.int64(53)),
 ('RYR2', 'USH2A', np.int64(51)),
 ('LRP1B', 'TP53', np.int64(49)),
 ('RYR2', 'ZFHX4', np.int64(49)),
 ('TP53', 'ZFHX4', np.int64(49))]

In [25]:
important_genes = [
    "TP53",
    "KRAS",
    "EGFR",
    "STK11",
    "KEAP1",
    "NF1",
    "BRAF",
    "PIK3CA",
    "ALK",
    "ROS1",
    "RET",
    "MET",
    "ERBB2",
    "SMARCA4",
    "RB1"
]

important_present = [
    g for g in important_genes
    if g in patient_gene.columns
]

important_present

['TP53',
 'KRAS',
 'EGFR',
 'STK11',
 'KEAP1',
 'NF1',
 'BRAF',
 'PIK3CA',
 'ALK',
 'ROS1',
 'RET',
 'MET',
 'ERBB2',
 'SMARCA4',
 'RB1']

In [26]:
patient_gene_cancer = patient_gene[
    important_present
]

co_matrix_cancer = (
    patient_gene_cancer.T
    .dot(patient_gene_cancer)
)

co_matrix_cancer

Hugo_Symbol,TP53,KRAS,EGFR,STK11,KEAP1,NF1,BRAF,PIK3CA,ALK,ROS1,RET,MET,ERBB2,SMARCA4,RB1
Hugo_Symbol,,,,,,,,,,,,,,,
TP53,107,25,22,11,14,21,12,10,17,8,9,5,4,5,8
KRAS,25,75,2,21,14,2,4,4,7,2,3,5,3,3,2
EGFR,22,2,35,0,3,2,4,2,4,2,1,2,0,1,3
STK11,11,21,0,40,13,5,2,1,4,0,1,1,1,4,1
KEAP1,14,14,3,13,40,5,3,2,5,0,0,3,1,4,1
NF1,21,2,2,5,5,30,1,2,6,2,0,1,2,3,4
BRAF,12,4,4,2,3,1,22,6,2,3,1,0,0,0,0
PIK3CA,10,4,2,1,2,2,6,17,3,1,2,1,0,0,1
ALK,17,7,4,4,5,6,2,3,23,4,1,1,1,2,1


In [27]:
freq = (
    patient_gene.sum(axis=0)
    .sort_values(ascending=False)
)

freq.head(50)

,0
Hugo_Symbol,
TTN,129
TP53,107
MUC16,99
RYR2,95
CSMD3,90
LRP1B,80
USH2A,79
KRAS,75
ZFHX4,71


In [28]:
important = [
    "TP53",
    "KRAS",
    "STK11",
    "KEAP1",
    "EGFR",
    "BRAF",
    "MET",
    "PIK3CA",
    "RB1"
]

patient_gene_imp = patient_gene[[
    g for g in important
    if g in patient_gene.columns
]]

co_imp = patient_gene_imp.T.dot(
    patient_gene_imp
)

co_imp

Hugo_Symbol,TP53,KRAS,STK11,KEAP1,EGFR,BRAF,MET,PIK3CA,RB1
Hugo_Symbol,,,,,,,,,
TP53,107,25,11,14,22,12,5,10,8
KRAS,25,75,21,14,2,4,5,4,2
STK11,11,21,40,13,0,2,1,1,1
KEAP1,14,14,13,40,3,3,3,2,1
EGFR,22,2,0,3,35,4,2,2,3
BRAF,12,4,2,3,4,22,0,6,0
MET,5,5,1,3,2,0,18,1,2
PIK3CA,10,4,1,2,2,6,1,17,1
RB1,8,2,1,1,3,0,2,1,10


In [29]:
patient_gene.to_csv(
    "/content/drive/MyDrive/Cancer Evolution Arena/Data/patient_gene_matrix.csv"
)

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/Cancer Evolution Arena/Data'